# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

I0000 00:00:1775427293.667730   17311 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775427293.734432   17311 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775427295.179530   17311 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 37s 0us/step


/home/deidarik/lab4ai/.venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


для себя - чтобы вычисления на гпу были

export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cuda_runtime/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cublas/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusolver/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusparse/lib


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [6]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [7]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(filters=channel_1, 
                                            kernel_size=(5, 5), 
                                            padding='same', 
                                            activation='relu')
        
        self.conv2 = tf.keras.layers.Conv2D(filters=channel_2, 
                                            kernel_size=(3, 3), 
                                            padding='same', 
                                            activation='relu')
        
        self.flatten = tf.keras.layers.Flatten()
        
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        
        x = self.conv2(x)
        
        x = self.flatten(x)
        
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [8]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [13]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [14]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every = 100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.8975512981414795, Accuracy: 10.9375, Val Loss: 2.8110625743865967, Val Accuracy: 13.799999237060547
Iteration 100, Epoch 1, Loss: 2.2342722415924072, Accuracy: 28.310644149780273, Val Loss: 1.8430132865905762, Val Accuracy: 39.79999923706055
Iteration 200, Epoch 1, Loss: 2.072460889816284, Accuracy: 32.190608978271484, Val Loss: 1.8418017625808716, Val Accuracy: 38.900001525878906
Iteration 300, Epoch 1, Loss: 1.9976222515106201, Accuracy: 33.918190002441406, Val Loss: 1.8689987659454346, Val Accuracy: 38.0
Iteration 400, Epoch 1, Loss: 1.930831789970398, Accuracy: 35.742671966552734, Val Loss: 1.7313464879989624, Val Accuracy: 42.0
Iteration 500, Epoch 1, Loss: 1.8875781297683716, Accuracy: 36.938621520996094, Val Loss: 1.6606366634368896, Val Accuracy: 43.29999923706055
Iteration 600, Epoch 1, Loss: 1.8560620546340942, Accuracy: 37.84577941894531, Val Loss: 1.6806344985961914, Val Accuracy: 41.900001525878906
Iteration 700, Epoch 1, Loss: 1.8302773237228

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [15]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every = 100


def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.3045530319213867, Accuracy: 12.5, Val Loss: 2.3941075801849365, Val Accuracy: 9.600000381469727
Iteration 100, Epoch 1, Loss: 1.9395527839660645, Accuracy: 30.97153663635254, Val Loss: 1.7243614196777344, Val Accuracy: 39.29999923706055
Iteration 200, Epoch 1, Loss: 1.7875443696975708, Accuracy: 36.66822052001953, Val Loss: 1.5025864839553833, Val Accuracy: 48.10000228881836
Iteration 300, Epoch 1, Loss: 1.6940556764602661, Accuracy: 40.01245880126953, Val Loss: 1.4391191005706787, Val Accuracy: 49.5
Iteration 400, Epoch 1, Loss: 1.6207913160324097, Accuracy: 42.60832214355469, Val Loss: 1.3807724714279175, Val Accuracy: 49.70000076293945
Iteration 500, Epoch 1, Loss: 1.5696367025375366, Accuracy: 44.3176155090332, Val Loss: 1.3235728740692139, Val Accuracy: 54.10000228881836
Iteration 600, Epoch 1, Loss: 1.5379722118377686, Accuracy: 45.37229537963867, Val Loss: 1.305397391319275, Val Accuracy: 52.89999771118164
Iteration 700, Epoch 1, Loss: 1.50814783573

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [16]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/home/deidarik/lab4ai/.venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.9283323287963867, Accuracy: 10.9375, Val Loss: 2.8951964378356934, Val Accuracy: 13.199999809265137
Iteration 100, Epoch 1, Loss: 2.230192184448242, Accuracy: 28.728342056274414, Val Loss: 1.8858109712600708, Val Accuracy: 38.70000076293945
Iteration 200, Epoch 1, Loss: 2.0740175247192383, Accuracy: 32.31498718261719, Val Loss: 1.8575892448425293, Val Accuracy: 38.80000305175781
Iteration 300, Epoch 1, Loss: 2.000197172164917, Accuracy: 34.08430099487305, Val Loss: 1.814969778060913, Val Accuracy: 39.39999771118164
Iteration 400, Epoch 1, Loss: 1.9287891387939453, Accuracy: 35.9375, Val Loss: 1.693314552307129, Val Accuracy: 41.60000228881836
Iteration 500, Epoch 1, Loss: 1.885388731956482, Accuracy: 37.00723648071289, Val Loss: 1.6679673194885254, Val Accuracy: 42.79999923706055
Iteration 600, Epoch 1, Loss: 1.855384111404419, Accuracy: 37.88217544555664, Val Loss: 1.6729133129119873, Val Accuracy: 43.5
Iteration 700, Epoch 1, Loss: 1.8290820121765137, Ac

Альтернативный менее гибкий способ обучения:

In [17]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - loss: 1.8079 - sparse_categorical_accuracy: 0.3908 - val_loss: 1.6922 - val_sparse_categorical_accuracy: 0.4280
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.6667 - sparse_categorical_accuracy: 0.4245


[1.6666916608810425, 0.4244999885559082]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [18]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every = 100


def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)


    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.3624792098999023, Accuracy: 4.6875, Val Loss: 2.307255983352661, Val Accuracy: 10.199999809265137
Iteration 100, Epoch 1, Loss: 2.1303813457489014, Accuracy: 22.493812561035156, Val Loss: 1.960145354270935, Val Accuracy: 31.0
Iteration 200, Epoch 1, Loss: 2.026320457458496, Accuracy: 27.246580123901367, Val Loss: 1.8366928100585938, Val Accuracy: 37.900001525878906
Iteration 300, Epoch 1, Loss: 1.9567480087280273, Accuracy: 30.263704299926758, Val Loss: 1.7581028938293457, Val Accuracy: 40.29999923706055
Iteration 400, Epoch 1, Loss: 1.8913471698760986, Accuracy: 33.0034294128418, Val Loss: 1.7093162536621094, Val Accuracy: 42.20000076293945
Iteration 500, Epoch 1, Loss: 1.8460971117019653, Accuracy: 34.746131896972656, Val Loss: 1.62580144405365, Val Accuracy: 45.5
Iteration 600, Epoch 1, Loss: 1.811133861541748, Accuracy: 36.083091735839844, Val Loss: 1.6064646244049072, Val Accuracy: 44.5
Iteration 700, Epoch 1, Loss: 1.7806661128997803, Accuracy: 37.28

In [19]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 1.6129 - sparse_categorical_accuracy: 0.4291 - val_loss: 1.3950 - val_sparse_categorical_accuracy: 0.4970
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.4136 - sparse_categorical_accuracy: 0.4941


[1.4136149883270264, 0.49410000443458557]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [21]:
device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'

def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [22]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.966892719268799, Accuracy: 6.25, Val Loss: 2.89778995513916, Val Accuracy: 12.5
Iteration 100, Epoch 1, Loss: 2.2431280612945557, Accuracy: 28.094058990478516, Val Loss: 1.929771900177002, Val Accuracy: 34.599998474121094
Iteration 200, Epoch 1, Loss: 2.0832302570343018, Accuracy: 31.887439727783203, Val Loss: 1.9189354181289673, Val Accuracy: 37.0
Iteration 300, Epoch 1, Loss: 2.0065650939941406, Accuracy: 33.80917739868164, Val Loss: 1.8912110328674316, Val Accuracy: 38.10000228881836
Iteration 400, Epoch 1, Loss: 1.9387046098709106, Accuracy: 35.54005432128906, Val Loss: 1.757503867149353, Val Accuracy: 42.099998474121094
Iteration 500, Epoch 1, Loss: 1.8929064273834229, Accuracy: 36.66728973388672, Val Loss: 1.6937410831451416, Val Accuracy: 40.70000076293945
Iteration 600, Epoch 1, Loss: 1.8634189367294312, Accuracy: 37.52859878540039, Val Loss: 1.7022156715393066, Val Accuracy: 39.70000076293945
Iteration 700, Epoch 1, Loss: 1.8381015062332153, Accur

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [24]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')
        self.bn1 = tf.keras.layers.BatchNormalization()
        
        
        self.conv2 = tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))
        
        
        self.conv3 = tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu')
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))
        
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.5) 
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, x, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.bn1(x, training=training)
        
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.pool1(x)
        
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = self.pool2(x)
        
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 4.099835395812988, Accuracy: 14.0625, Val Loss: 2.3560938835144043, Val Accuracy: 11.300000190734863
Iteration 700, Epoch 1, Loss: 1.6767830848693848, Accuracy: 40.24385070800781, Val Loss: 1.1731696128845215, Val Accuracy: 59.500003814697266
Iteration 1400, Epoch 2, Loss: 1.2520899772644043, Accuracy: 55.66682815551758, Val Loss: 0.9882185459136963, Val Accuracy: 65.5
Iteration 2100, Epoch 3, Loss: 1.069759726524353, Accuracy: 62.31875991821289, Val Loss: 0.9325389266014099, Val Accuracy: 67.79999542236328
Iteration 2800, Epoch 4, Loss: 0.942430317401886, Accuracy: 67.09430694580078, Val Loss: 0.8090838193893433, Val Accuracy: 72.0999984741211
Iteration 3500, Epoch 5, Loss: 0.8402244448661804, Accuracy: 70.67005157470703, Val Loss: 0.7765854001045227, Val Accuracy: 74.5999984741211
Iteration 4200, Epoch 6, Loss: 0.7762048840522766, Accuracy: 72.78048706054688, Val Loss: 0.7253649830818176, Val Accuracy: 74.69999694824219
Iteration 4900, Epoch 7, Loss: 0.685

Опишите все эксперименты, результаты. Сделайте выводы.

## Список экспериментов
* Эксперимент №1 - использовалась простая 3 - слойная CNN (из предыдущих пунктов). Результат: ~58% точности. Проблема: медленная сходимость и низкая обобщающая способность.
* Эксперимент №2 - увеличил количество фильтров до 128 и добавил слои MaxPooling2D. Результат: точность выросла до 64%, но началось переобучение.
* Эксперимент №3 -  в архитектуру добавил слои BatchNormalization после каждой свертки и Dropout = 0.5 перед финальным слоем, также использовал оптимизатор Adam с lr=1e-3. Результат: точность превысила 78% за 10 эпох.
